### Import Check ⬇️

In [1]:
# Import Check
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Alle Pakete erfolgreich geladen ✅")

Alle Pakete erfolgreich geladen ✅


### ROM-Struktur verstehen, welche Version ist vorhanden? ⬇️

In [2]:
ROM_PATH = "../data/pokemon_firered.gba"

with open(ROM_PATH, "rb") as f:
    header = f.read(0xC0)

game_code = header[0xAC:0xB0].decode("ascii")
game_title = header[0xA0:0xAC].decode("ascii").strip()

print(f"Spieltitel: {game_title}")
print(f"Game Code:  {game_code}")

Spieltitel: POKEMON FIRE
Game Code:  BPRD


### Pokemon Zeichenkodierung Hexa nicht Standard: Quelle -> Claude.ai ⬇️

In [3]:
CHAR_MAP = {
    0xBB: "A", 0xBC: "B", 0xBD: "C", 0xBE: "D", 0xBF: "E",
    0xC0: "F", 0xC1: "G", 0xC2: "H", 0xC3: "I", 0xC4: "J",
    0xC5: "K", 0xC6: "L", 0xC7: "M", 0xC8: "N", 0xC9: "O",
    0xCA: "P", 0xCB: "Q", 0xCC: "R", 0xCD: "S", 0xCE: "T",
    0xCF: "U", 0xD0: "V", 0xD1: "W", 0xD2: "X", 0xD3: "Y",
    0xD4: "Z",
    0xD5: "a", 0xD6: "b", 0xD7: "c", 0xD8: "d", 0xD9: "e",
    0xDA: "f", 0xDB: "g", 0xDC: "h", 0xDD: "i", 0xDE: "j",
    0xDF: "k", 0xE0: "l", 0xE1: "m", 0xE2: "n", 0xE3: "o",
    0xE4: "p", 0xE5: "q", 0xE6: "r", 0xE7: "s", 0xE8: "t",
    0xE9: "u", 0xEA: "v", 0xEB: "w", 0xEC: "x", 0xED: "y",
    0xEE: "z",
    0x00: " ", 0xFF: ""  # Leerzeichen & String-Ende
}

### Offset Testausgabe aller Pokemon mit 10 Bytes Abstand ⬇️

In [4]:
def decode_pokemon_string(data):
    result = ""
    for byte in data:
        if byte == 0xFF:  # String-Ende
            break
        result += CHAR_MAP.get(byte, "?")
    return result.strip()

# Anpassbare Variablen festlegen und Namen auslesen
NAMES_OFFSET = 0x245EE0
POKEMON_COUNT = 151
NAME_LENGTH = 10

pokemon_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = NAMES_OFFSET + (i * NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        pokemon_namen.append({"id": i + 1, "name": name})

for p in pokemon_namen:
    print(f"#{p['id']:03d} {p['name']}")

#001 SANDAM
#002 ER
#003 AN?
#004 RINA
#005 OQUEEN
#006 DORAN?
#007 IDORINO
#008 NIDOKING
#009 PIEPI
#010 PIXI
#011 VULPIX
#012 VULNON
#013 A
#014 LUFF
#015 DELUFF
#016 AT
#017 LBAT
#018 YRAPLA
#019 DUFLOR
#020 GIFLOR
#021 PARAS
#022 PARASEK
#023 
#024 
#025 DIGD
#026 A
#027 DRI
#028 UZI
#029 NOBILIKAT
#030 ENTON
#031 ENTORON
#032 MENKI
#033 RASAFF
#034 FUKANO
#035 
#036 I
#037 SEL
#038 PUTZI
#039 APPO
#040 BRA
#041 KADABRA
#042 SIMSALA
#043 MACHOLLO
#044 
#045 K
#046 EI
#047 NSA
#048 IGARIA
#049 ZENIA
#050 NTACHA
#051 ENTOXA
#052 KLEINSTEIN
#053 
#054 GEOWAZ
#055 PONITA
#056 GALLOP
#057 A
#058 ON
#059 US
#060 NETILO
#061 GNETON
#062 ORENTA
#063 DODU
#064 DODRI
#065 JUROB
#066 JUGONG
#067 SLEIMA
#068 
#069 OK
#070 HAS
#071 TOS
#072 BULAK
#073 LPOLLO
#074 GENGAR
#075 ONIX
#076 TRAUMATO
#077 
#078 KRABBY
#079 
#080 ER
#081 OBAL
#082 TROBAL
#083 EI
#084 OKOWEI
#085 TRAGOSSO
#086 KNOGGA
#087 KICKLEE
#088 NOCKCHA
#089 N
#090 P
#091 N
#092 MOG
#093 ORN
#094 ZEROS
#095 HANEIRA
#096 TANGELA
#0

### Offensichtlich falscher Offset in vorheriger Zelle
### Suche nach "BISASAM" (Pokemon 001)
### In Pokemon-Kodierung übersetzen ⬇️

In [5]:
CHAR_MAP_REVERSE = {v: k for k, v in CHAR_MAP.items()}

def encode_pokemon_string(text):
    return bytes([CHAR_MAP_REVERSE.get(c, 0x00) for c in text])

# Nach "BISASAM" in der ROM suchen
search = encode_pokemon_string("BISASAM")
print(f"Suche nach Bytes: {search.hex()}")

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

position = rom_data.find(search)
print(f"Gefunden an Position: {hex(position)}")

# Neuer Offset = Position von BISASAM
# (da BISASAM Pokémon #001 ist, ist das direkt der Basis-Offset)
print(f"Neuer NAMES_OFFSET: {hex(position)}")

Suche nach Bytes: bcc3cdbbcdbbc7
Gefunden an Position: 0x1907b7
Neuer NAMES_OFFSET: 0x1907b7


### Neuen Offset testen ⬇️

In [6]:
NAMES_OFFSET = 0x1907b7
POKEMON_COUNT = 151
NAME_LENGTH = 10

pokemon_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = NAMES_OFFSET + (i * NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        pokemon_namen.append({"id": i + 1, "name": name})

for p in pokemon_namen:
    print(f"#{p['id']:03d} {p['name']}")

#001 BISASAM so
#002 ll es sein
#003 ??Es ist l
#004 eicht zu t
#005 rainieren?
#006 ???? du m?
#007 chtest als
#008 o das?PFLA
#009 NZEN?POK?M
#010 ON BISASAM
#011 ?
#012 OK?MON ist
#013 wirklich?
#014 energiegel
#015 aden?
#016 rh?lt????
#017 PROF? EICH
#018 ? Wenn du
#019 auf ein wi
#020 ldes?POK?M
#021 ON triffst
#022 ? dann kan
#023 n dein?POK
#024 ?MON dageg
#025 en k?mpfen
#026 ?
#027 CH? ??? we
#028 nn du dein
#029 ?POK?MON k
#030 ?mpfen l?s
#031 st? wird e
#032 s?st?rker?
#033 
#034 H? Hallo?
#035 ????Wie ge
#036 ht es dem
#037 POK?MON??E
#038 s scheint
#039 dich sehr
#040 zu m?gen??
#041 Du musst a
#042 ls POK?MON
#043 TRAINER s
#044 ehr?talent
#045 iert sein?
#046 ?Du hast e
#047 twas f?r m
#048 ich?
#049 ergibt?PRO
#050 F? EICH da
#051 s PAKET?
#052 h? Auf die
#053 sen SPEZIA
#054 L?POK?BALL
#055 ?warte ich
#056 schon lan
#057 ge? ?Viele
#058 n Dank?
#059 OF? EICH?
#060 Ach ja? Ic
#061 h habe ein
#062 e?gro?e Au
#063 fgabe f?r
#064 euch?
#065 dem Tisch
#066 dort seht
#067 i

### Falscher Offset mit "BISASAM", wahrscheinlich einfach nur Storytext
### Suche eingrenzen mit "BISASAM" und "BISAKNOSP" ⬇️

In [7]:
search1 = encode_pokemon_string("BISASAM")
search2 = encode_pokemon_string("BISAKNOSP")

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

# Alle Vorkommen von BISASAM finden
pos = 0
while True:
    position = rom_data.find(search1, pos)
    if position == -1:
        break

    # Prüfen ob BISAKNOSP genau 10 Bytes später kommt
    next_pos = position + 10
    if rom_data[next_pos:next_pos + len(search2)] == search2:
        print(f"✅ Namenstabelle gefunden bei: {hex(position)}")
        break
    else:
        print(f"❌ Falscher Treffer bei: {hex(position)}")

    pos = position + 1

❌ Falscher Treffer bei: 0x1907b7
❌ Falscher Treffer bei: 0x190814
❌ Falscher Treffer bei: 0x245dbb


### Falsche Treffer in Zelle zuvor.
### Prüfen was 10 Bytes später steht.
### Vielleicht doch anderer Bytes Absatnd? ⬇️

In [8]:
search1 = encode_pokemon_string("BISASAM")

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

pos = 0
while True:
    position = rom_data.find(search1, pos)
    if position == -1:
        break

    # Zeige was nach BISASAM kommt (20 Bytes)
    after = rom_data[position:position + 30]
    decoded = decode_pokemon_string(after)

    print(f"Position {hex(position)}: '{decoded}'")
    print(f"  Raw Bytes: {after.hex()}")
    print()

    pos = position + 1

Position 0x1907b7: 'BISASAM soll es sein??Es ist l'
  Raw Bytes: bcc3cdbbcdbbc700e7e3e0e000d9e700e7d9dde2abfebfe700dde7e800e0

Position 0x190814: 'BISASAM?'
  Raw Bytes: bcc3cdbbcdbbc7acffbeddd9e7d9e700cac9c51bc7c9c800dde7e800ebdd

Position 0x245dbb: 'BISASAM'
  Raw Bytes: bcc3cdbbcdbbc7ff000000bcc3cdbbc5c8c9cdcaff00bcc3cdbbc0c6c9cc



### Position 0x245dbb: 'BISASAM' sieht vielversprechend aus ff = String Ende dann 3 Nullbytes Abstand
### bcc3cdbbcdbbc7ff000000 = (B I S A S A M = 7 Bytes) + (ff = 1 Bytes) + (00 00 00 = 3 Bytes) also 7+1+3=11 Bytes
### Gegentest: bcc3cdbbc5c8c9cdcaff00 = (B I S A K N O S P = 9 Bytes) + (ff = 1 Bytes) + (00 = 1 Bytes) also 9+1+1=11 Bytes
### Es sind nicht 10 Bytes Abstand sondern 11
### Offset Test mit neuen Erkenntnissen ⬇️

In [9]:
NAMES_OFFSET = 0x245dbb
POKEMON_COUNT = 151
NAME_LENGTH = 11  # ← geändert!

pokemon_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = NAMES_OFFSET + (i * NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        pokemon_namen.append({"id": i + 1, "name": name})

df_namen = pd.DataFrame(pokemon_namen)
df_namen

,id,name
0,1,BISASAM
1,2,BISAKNOSP
2,3,BISAFLOR
3,4,GLUMANDA
4,5,GLUTEXO
...,...,...
146,147,DRATINI
147,148,DRAGONIR
148,149,DRAGORAN
149,150,MEWTU


### Offset für Pokenamen richtig identifiziert
### Jetzt Basiswerte also HP, ATK, DEF, Speed, SP.ATK, SP.DEF (45, 49, 49, 45, 65, 65) extrahieren ⬇️


In [10]:
import struct
bisasam_stats = bytes([45, 49, 49, 45, 65, 65])

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

pos = 0
while True:
    position = rom_data.find(bisasam_stats, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    print(f"Raw Bytes: {rom_data[position:position+28].hex()}")
    print()

    pos = position + 1

Treffer bei: 0x2546c4
Raw Bytes: 2d31312d41410c032d400001000000001f1446030107410000030000

Nichts weiter gefunden!


### Mit aller Wahrscheinlichkeit Treffer
### 2d 31 31 2d 41 41 = sind Bisasams Werte (45 49 49 45 65 65)
### Alle stats bei allen Pokemon auslesen ⬇️

In [11]:
STATS_OFFSET = 0x2546c4
POKEMON_COUNT = 151

pokemon_stats = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = STATS_OFFSET + (i * 28)
        f.seek(offset)
        data = f.read(28)

        stats = struct.unpack_from("BBBBBB", data, 0)

        pokemon_stats.append({
            "hp":             stats[0],
            "angriff":        stats[1],
            "verteidigung":   stats[2],
            "speed":          stats[3],
            "sp_angriff":     stats[4],
            "sp_verteidigung":stats[5],
        })

df_stats = pd.DataFrame(pokemon_stats)
df_stats

,hp,angriff,verteidigung,speed,sp_angriff,sp_verteidigung
0,45,49,49,45,65,65
1,60,62,63,60,80,80
2,80,82,83,80,100,100
3,39,52,43,65,60,50
4,58,64,58,80,80,65
...,...,...,...,...,...,...
146,41,64,45,50,50,50
147,61,84,65,70,70,70
148,91,134,95,80,100,100
149,106,110,90,130,154,90


### Stats ausgelesen
### Typen sollten im selben Byte-Block sein direkt hinter den stats (Check = Bisasam 001/00 sollte Gras/Gift sein) ⬇️

In [12]:
# Typ-ID
TYPES = {
    0:  "Normal",  1:  "Kampf",   2:  "Flug",
    3:  "Gift",    4:  "Boden",   5:  "Gestein",
    6:  "Käfer",   7:  "Geist",   8:  "Stahl",
    10: "Feuer",   11: "Wasser",  12: "Gras",
    13: "Elektro", 14: "Psycho",  15: "Eis",
    16: "Drache",  17: "Unlicht"
}

pokemon_typen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(POKEMON_COUNT):
        offset = STATS_OFFSET + (i * 28)
        f.seek(offset)
        data = f.read(28)

        stats = struct.unpack_from("BBBBBBBB", data, 0)

        pokemon_typen.append({
            "typ1": TYPES.get(stats[6], f"Unbekannt({stats[6]})"),
            "typ2": TYPES.get(stats[7], f"Unbekannt({stats[7]})")
        })

df_typen = pd.DataFrame(pokemon_typen)
df_typen

,typ1,typ2
0,Gras,Gift
1,Gras,Gift
2,Gras,Gift
3,Feuer,Feuer
4,Feuer,Feuer
...,...,...
146,Drache,Drache
147,Drache,Drache
148,Drache,Flug
149,Psycho,Psycho


### Typen wurden erfolgreich extrahiert
### Jetzt Zusammenführung von drei df zu einem df (pd.concat weil alle df gleich viele Zeilen, Reihenfolge) ⬇️

In [13]:
df_pokemon = pd.concat([df_namen, df_stats, df_typen], axis=1)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df_pokemon = df_pokemon.set_index("id")
df_pokemon

,name,hp,angriff,verteidigung,speed,sp_angriff,sp_verteidigung,typ1,typ2
id,,,,,,,,,
1,BISASAM,45,49,49,45,65,65,Gras,Gift
2,BISAKNOSP,60,62,63,60,80,80,Gras,Gift
3,BISAFLOR,80,82,83,80,100,100,Gras,Gift
4,GLUMANDA,39,52,43,65,60,50,Feuer,Feuer
5,GLUTEXO,58,64,58,80,80,65,Feuer,Feuer
...,...,...,...,...,...,...,...,...,...
147,DRATINI,41,64,45,50,50,50,Drache,Drache
148,DRAGONIR,61,84,65,70,70,70,Drache,Drache
149,DRAGORAN,91,134,95,80,100,100,Drache,Flug


### Merge von allen df + .set_index("id") = id Spalte als Index gesetzt
### Nun Move-Daten also Angriffe extrahieren
### Auch bei Move-Daten muss Offset gefunden werden
### Vorgeschlagene Reihenfolge laut Claude.ai = 10, 95, 100, 15
### Move Flammenwurf bekannte Werte = Typ -> Feuer / Stärke -> 95 / Genauigkeit -> 100 / PowerPoints -> 15 ⬇️

In [14]:
# Flammenwurf Werte als Bytes
flammenwurf = bytes([10, 95, 100, 15])

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

pos = 0
while True:
    position = rom_data.find(flammenwurf, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    print(f"Raw Bytes: {rom_data[position:position+9].hex()}")
    print()

    pos = position + 1

Nichts weiter gefunden!


### Nichts gefunden, vielleicht andere Reihenfolge der Bytes?
### Teste vorerst nur Stärke und Genauigkeit ⬇️

In [15]:
# Nur Stärke (95) und Genauigkeit (100) suchen
flammenwurf = bytes([95, 10, 100])  # Stärke, Typ, Genauigkeit

with open("../data/pokemon_firered.gba", "rb") as f:
    rom_data = f.read()

pos = 0
while True:
    position = rom_data.find(flammenwurf, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    print(f"Raw Bytes: {rom_data[position-1:position+8].hex()}")
    print()

    pos = position + 1

Treffer bei: 0x250da5
Raw Bytes: 045f0a640f0a000012

Nichts weiter gefunden!


### Erfolg! RAW Byte Analyse = 04 5f 0a 64 0f 0a 00 00 12 bedeutet 04 -> Effekt-ID / 5f -> 95=Stärke / 0a -> 10=Typ_Feuer / 64 -> 100=Genauigkeit / 0f -> 15=PowerPoints
### Struktur ist also == Effekt | Stärke | Typ | Genauigkeit | PP | ...
### Der Move-Offset ist dann in diesem Fall 0x250da5 - 1 = 0x250da4 <-- Wieso -1? Weil zuerst nach Stärke gesucht, Effekt kommt jedoch vor Stärke, bedeutet: Move-Block beginnt 1 Byte früher bei 0x250da4
### Weiter mit alle Moves auslesen ⬇️

In [16]:
MOVES_OFFSET = 0x250da4
MOVE_COUNT = 165

moves_data = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(MOVE_COUNT):
        offset = MOVES_OFFSET + (i * 9)
        f.seek(offset)
        data = f.read(9)

        stats = struct.unpack_from("BBBBB", data, 0)

        moves_data.append({
            "move_id":     i + 1,
            "staerke":     stats[1],
            "typ":         TYPES.get(stats[2], f"Unbekannt({stats[2]})"),
            "genauigkeit": stats[3],
            "pp":          stats[4],
        })

df_moves = pd.DataFrame(moves_data)
df_moves

,move_id,staerke,typ,genauigkeit,pp
0,1,95,Feuer,100,15
1,2,0,Normal,46,0
2,3,0,Stahl,0,0
3,4,25,Normal,0,0
4,5,120,Wasser,80,5
...,...,...,...,...,...
160,161,40,Normal,100,15
161,162,0,Normal,109,0
162,163,0,Normal,0,0
163,164,15,Normal,0,0


### 6.1% Typ unbekannt. Kurzer Check ob das so passt ⬇️

In [17]:
unbekannt = df_moves[df_moves["typ"].str.contains("Unbekannt")]
unbekannt

,move_id,staerke,typ,genauigkeit,pp
6,7,0,Unbekannt(50),0,0
10,11,0,Unbekannt(18),0,0
14,15,0,Unbekannt(50),0,0
18,19,0,Unbekannt(51),0,0
22,23,0,Unbekannt(51),0,0
26,27,0,Unbekannt(18),0,0
30,31,0,Unbekannt(50),0,0
34,35,0,Unbekannt(22),0,0
38,39,0,Unbekannt(22),0,0
42,43,0,Unbekannt(18),0,0


### Kurzer Check mit Claude.ai zeigt ist normal
### Daten werden angepasst in Data Cleaning später
### Jetzt erstmal weiter alle Moves extrahieren
### Auch hier vorerst Offset finden mit Move Flammenwurf ⬇️

In [18]:
search = encode_pokemon_string("FLAMMENWURF")
print(f"Suche nach Bytes: {search.hex()}")

pos = 0
while True:
    position = rom_data.find(search, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    pos = position + 1

Suche nach Bytes: c0c6bbc7c7bfc8d1cfccc0
Treffer bei: 0x247215
Nichts weiter gefunden!


### Ein Treffer! -> Sehr eindeutig.
### Jetzt Struktur prüfen
### Vermutung das Move-Namen 13 Bytes lang sind ⬇️

In [19]:
position = 0x247215

print(f"Gefunden bei: {hex(position)}")
print(f"Raw Bytes: {rom_data[position:position+30].hex()}")

# Dekodiert anzeigen
for i in range(5):
    name = decode_pokemon_string(rom_data[position + (i * 13):position + (i * 13) + 13])
    print(f"Name {i}: '{name}'")

Gefunden bei: 0x247215
Raw Bytes: c0c6bbc7c7bfc8d1cfccc0ff00d1bfc3cdcdc8bfbcbfc6ff0000bbcbcfbb
Name 0: 'FLAMMENWURF'
Name 1: 'WEISSNEBEL'
Name 2: 'AQUAKNARRE'
Name 3: 'HYDROPUMPE'
Name 4: 'SURFER'


### Passt, Namen sind 13 Bytes lang
### Jetzt Basis Offset berechnen
### Flammenwurf ist Move 52 also unter Index 51 zu finden
### Somit müssen wir 0x247215 - (51 * 13) rechnen ⬇️

In [20]:
flammenwurf_offset = 0x247215
MOVE_NAME_LENGTH = 13
MOVES_NAMES_OFFSET = flammenwurf_offset - (51 * MOVE_NAME_LENGTH)

print(f"Moves Namen Offset: {hex(MOVES_NAMES_OFFSET)}")

# Alle Move-Namen auslesen
move_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(MOVE_COUNT):
        offset = MOVES_NAMES_OFFSET + (i * MOVE_NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(MOVE_NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        move_namen.append({"move_id": i + 1, "move_name": name})

df_move_namen = pd.DataFrame(move_namen).set_index("move_id")
df_move_namen.head(10)

Moves Namen Offset: 0x246f7e


,move_name
move_id,
1,KARATESCHLAG
2,DUPLEXHIEB
3,KOMETENHIEB
4,MEGAHIEB
5,ZAHLTAG
6,FEUERSCHLAG
7,EISHIEB
8,DONNERSCHLAG
9,KRATZER


### Karateschlag sollte laut Infos Move 2 sein
### Bedeutet: Offset ist um einen Move verschoben
### Flammenwurf ist Move 53 und nicht wie zuvor angenommen 52
### Anpassung von Flammenwurf für Offset Fund ⬇️

In [21]:
MOVES_NAMES_OFFSET = 0x247215 - (52 * MOVE_NAME_LENGTH)

print(f"Moves Namen Offset: {hex(MOVES_NAMES_OFFSET)}")

move_namen = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(MOVE_COUNT):
        offset = MOVES_NAMES_OFFSET + (i * MOVE_NAME_LENGTH)
        f.seek(offset)
        name_bytes = f.read(MOVE_NAME_LENGTH)
        name = decode_pokemon_string(name_bytes)
        move_namen.append({"move_id": i + 1, "move_name": name})

df_move_namen = pd.DataFrame(move_namen).set_index("move_id")
df_move_namen.head(10)

Moves Namen Offset: 0x246f71


,move_name
move_id,
1,PFUND
2,KARATESCHLAG
3,DUPLEXHIEB
4,KOMETENHIEB
5,MEGAHIEB
6,ZAHLTAG
7,FEUERSCHLAG
8,EISHIEB
9,DONNERSCHLAG


### Anscheinend ebenfalls falsche Info mit Index von Flammenwurf
### Ruckler soll Move 1 sein und damit der Basis-Offset
### Versuche Direkt nach Ruckler zu suchen = cccfbdc5c6bfcc

In [22]:
search = encode_pokemon_string("RUCKLER")
print(f"Suche nach Bytes: {search.hex()}")

pos = 0
while True:
    position = rom_data.find(search, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    pos = position + 1

Suche nach Bytes: cccfbdc5c6bfcc
Nichts weiter gefunden!


### Nichts gefunden
### Nach Recherche im Pokewiki https://www.pokewiki.de/Attacken_der_1._Generation wurde klar das Pfund richtig war
### Als nächstes df_move_namen und df_moves zusammenführen ⬇️

In [23]:
df_moves_komplett = pd.concat([df_move_namen, df_moves.set_index("move_id")], axis=1)
df_moves_komplett

,move_name,staerke,typ,genauigkeit,pp
move_id,,,,,
1,PFUND,95,Feuer,100,15
2,KARATESCHLAG,0,Normal,46,0
3,DUPLEXHIEB,0,Stahl,0,0
4,KOMETENHIEB,25,Normal,0,0
5,MEGAHIEB,120,Wasser,80,5
...,...,...,...,...,...
161,TRIPLETTE,40,Normal,100,15
162,SUPERZAHN,0,Normal,109,0
163,SCHLITZER,0,Normal,0,0


### df_moves wurde noch nicht angepasst mit -1 Offset wie bei df_move_namen
### Also auch hier noch Basis-Offset anpassen ⬇️

In [24]:
MOVES_OFFSET = 0x250da4 - (52 * 9)  # 9 Bytes pro Move

print(f"Neuer MOVES_OFFSET: {hex(MOVES_OFFSET)}")

Neuer MOVES_OFFSET: 0x250bd0


In [25]:
MOVES_OFFSET = 0x250bd0
MOVE_COUNT = 165

moves_data = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(MOVE_COUNT):
        offset = MOVES_OFFSET + (i * 9)
        f.seek(offset)
        data = f.read(9)

        stats = struct.unpack_from("BBBBB", data, 0)

        moves_data.append({
            "move_id":     i + 1,
            "staerke":     stats[1],
            "typ":         TYPES.get(stats[2], f"Unbekannt({stats[2]})"),
            "genauigkeit": stats[3],
            "pp":          stats[4],
        })

df_moves = pd.DataFrame(moves_data)
df_moves

,move_id,staerke,typ,genauigkeit,pp
0,1,0,Normal,0,30
1,2,0,Normal,0,50
2,3,0,Unbekannt(51),0,0
3,4,35,Normal,0,0
4,5,60,Flug,100,35
...,...,...,...,...,...
160,161,0,Psycho,80,15
161,162,0,Normal,157,0
162,163,0,Unbekannt(24),0,0
163,164,20,Normal,0,0


### Stimmt noch nicht ganz
### Kurzer Strukturtest, zeige 20 Bytes ab Flammenwurf Position ⬇️

In [26]:
position = 0x250da4 + (52 * 9)  # Flammenwurf mit aktuellem Offset
print(f"Flammenwurf bei: {hex(position)}")

with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(position)
    raw = f.read(20)

print(f"Raw Byte: {raw.hex()}")

Flammenwurf bei: 0x250f78
Raw Byte: 210003550a640000160000004c320e64190a0000


In [27]:
print(hex(0x250da4 - (52 * 9)))

0x250bd0


In [28]:
with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(0x250bd0)
    raw = f.read(9)

print(f"Move #1 Raw: {raw.hex()}")
print(f"Stärke:      {raw[1]}")
print(f"Typ:         {raw[2]}")
print(f"Genauigkeit: {raw[3]}")
print(f"PP:          {raw[4]}")

Move #1 Raw: 320000001e00100008
Stärke:      0
Typ:         0
Genauigkeit: 0
PP:          30


### Die Werte stimmen nicht für Pfund
### Das Bedeutet Flammenwurf ist nicht Index 52
### Tipp von claude.ai: In Pokemon gibt es oft leeren Move 0, bedeutet Index 53 statt 52? ⬇️

In [29]:
MOVES_OFFSET = 0x250da4 - (53 * 9)
print(hex(MOVES_OFFSET))

with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(MOVES_OFFSET + 9)  # Index 1 = Pfund
    raw = f.read(9)

print(f"Pfund Raw:   {raw.hex()}")
print(f"Stärke:      {raw[1]}")
print(f"Typ:         {raw[2]}")
print(f"Genauigkeit: {raw[3]}")
print(f"PP:          {raw[4]}")

0x250bc7
Pfund Raw:   320000001e00100008
Stärke:      0
Typ:         0
Genauigkeit: 0
PP:          30


### Stimmt auch nicht, Werte sollten 40/0/100/35 sein
### Suche direkt nach Pfund Werte ⬇️

In [30]:
pfund = bytes([40, 0, 100, 35])

pos = 0
while True:
    position = rom_data.find(pfund, pos)
    if position == -1:
        print("Nichts weiter gefunden!")
        break

    print(f"Treffer bei: {hex(position)}")
    print(f"Raw Bytes: {rom_data[position-1:position+8].hex()}")
    print()

    pos = position + 1

Treffer bei: 0x250b35
Raw Bytes: 002800642300000033

Treffer bei: 0x250ba1
Raw Bytes: 002800642300000033

Nichts weiter gefunden!


### Zwei Treffer mit identischen Bytes
### Prüfen ob Werte von Karateschlag (Move 2) direkt dahinter kommen ⬇️

In [31]:
# Treffer 1
with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(0x250b35)
    raw = f.read(9 * 3)

for i in range(3):
    block = raw[i*9:(i+1)*9]
    print(f"Move #{i+1}: {block.hex()}")
    print(f"  Stärke: {block[1]}, Typ: {block[2]}, Gen: {block[3]}, PP: {block[4]}")

Move #1: 280064230000003300
  Stärke: 0, Typ: 100, Gen: 35, PP: 0
Move #2: 00002b320164190000
  Stärke: 0, Typ: 43, Gen: 50, PP: 1
Move #3: 00330000001d0f0055
  Stärke: 51, Typ: 0, Gen: 0, PP: 0


In [32]:
# Treffer 2
with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(0x250ba1)
    raw = f.read(9 * 3)

for i in range(3):
    block = raw[i*9:(i+1)*9]
    print(f"Move #{i+1}: {block.hex()}")
    print(f"  Stärke: {block[1]}, Typ: {block[2]}, Gen: {block[3]}, PP: {block[4]}")

Move #1: 280064230000003300
  Stärke: 0, Typ: 100, Gen: 35, PP: 0
Move #2: 0000003700641e0000
  Stärke: 0, Typ: 0, Gen: 55, PP: 0
Move #3: 00330000002601001e
  Stärke: 51, Typ: 0, Gen: 0, PP: 0


In [33]:
with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(0x250bd0)
    raw = f.read(9)

print(f"Stärke: {raw[1]}, Typ: {raw[2]}, Gen: {raw[3]}, PP: {raw[4]}")

Stärke: 0, Typ: 0, Gen: 0, PP: 30


In [34]:
with open("../data/pokemon_firered.gba", "rb") as f:
    f.seek(0x250bd0)
    raw = f.read(9)

print(f"Raw:         {raw.hex()}")
print(f"Effekt:      {raw[0]}")
print(f"Stärke:      {raw[1]}")  # stats[1] nicht stats[0]!
print(f"Typ:         {raw[2]}")
print(f"Genauigkeit: {raw[3]}")
print(f"PP:          {raw[4]}")

Raw:         320000001e00100008
Effekt:      50
Stärke:      0
Typ:         0
Genauigkeit: 0
PP:          30


### Werte stimmen nicht mit den vorgegebenen Werten überein
### Pfund gefunden bei 0x250b35 (aber Block startet 1 Byte früher bei 0x250b34)
### Flammenwurf gefunden bei 0x250da4
### Weiterer Test = Differenz zwischen Pfund und Flammenwurf ausrechnen, sollte die echte Block-Größe verraten ⬇️

In [35]:
pfund_pos = 0x250b34      # Effekt-Byte von Pfund
flammen_pos = 0x250da4    # Effekt-Byte von Flammenwurf

differenz = flammen_pos - pfund_pos
print(f"Differenz in Bytes: {differenz}")
print(f"Anzahl Moves dazwischen: 52")
print(f"Bytes pro Move: {differenz / 52}")

Differenz in Bytes: 624
Anzahl Moves dazwischen: 52
Bytes pro Move: 12.0


### 12 Bytes pro Move sind richtig! Nicht 9
### Pfund ist Move #1 -> Index 0
### Flammenwurf ist Move #53 -> Index 52
### Pfund Effekt-Byte ist bei 0x250b34
### Jetzt alles neu berechnen

In [36]:
MOVES_OFFSET = 0x250b34
MOVE_COUNT = 165

moves_data = []

with open("../data/pokemon_firered.gba", "rb") as f:
    for i in range(MOVE_COUNT):
        offset = MOVES_OFFSET + (i * 12)
        f.seek(offset)
        data = f.read(12)

        moves_data.append({
            "move_id":     i + 1,
            "effekt":      data[0],
            "staerke":     data[1],
            "typ":         TYPES.get(data[2], f"Unbekannt({data[2]})"),
            "genauigkeit": data[3],
            "pp":          data[4],
        })

df_moves = pd.DataFrame(moves_data)
df_moves

,move_id,effekt,staerke,typ,genauigkeit,pp
0,1,0,40,Normal,100,35
1,2,43,50,Kampf,100,25
2,3,29,15,Normal,85,10
3,4,29,18,Normal,85,15
4,5,0,80,Normal,85,20
...,...,...,...,...,...,...
160,161,36,80,Normal,100,10
161,162,40,1,Normal,90,10
162,163,43,70,Normal,100,20
163,164,79,0,Normal,0,10


### Daten der vorherigen Zelle sehen nun richtig aus
### Jetzt wieder set_index("move_id") und alles zusammenführen mit concat ⬇️

In [37]:
df_moves = df_moves.set_index("move_id")
df_moves_komplett = pd.concat([df_move_namen, df_moves], axis=1)
df_moves_komplett

,move_name,effekt,staerke,typ,genauigkeit,pp
move_id,,,,,,
1,PFUND,0,40,Normal,100,35
2,KARATESCHLAG,43,50,Kampf,100,25
3,DUPLEXHIEB,29,15,Normal,85,10
4,KOMETENHIEB,29,18,Normal,85,15
5,MEGAHIEB,0,80,Normal,85,20
...,...,...,...,...,...,...
161,TRIPLETTE,36,80,Normal,100,10
162,SUPERZAHN,40,1,Normal,90,10
163,SCHLITZER,43,70,Normal,100,20


### df_moves_komplett erfolgreich zusammengeführt
### Rohdaten als CSV Speichern ⬇️

In [38]:
df_pokemon.to_csv("../data/pokemon.csv")

In [39]:
df_moves_komplett.to_csv("../data/moves.csv")

In [40]:
import os

for file in ["pokemon.csv", "moves.csv"]:
    path = f"../data/{file}"
    size = os.path.getsize(path)
    print(f"{file}: {size} Bytes ")

pokemon.csv: 6641 Bytes 
moves.csv: 5459 Bytes 


### CSV anzeigen ⬇️

In [41]:
df_pokemon = pd.read_csv("../data/pokemon.csv", index_col="id")

df_pokemon

,name,hp,angriff,verteidigung,speed,sp_angriff,sp_verteidigung,typ1,typ2
id,,,,,,,,,
1,BISASAM,45,49,49,45,65,65,Gras,Gift
2,BISAKNOSP,60,62,63,60,80,80,Gras,Gift
3,BISAFLOR,80,82,83,80,100,100,Gras,Gift
4,GLUMANDA,39,52,43,65,60,50,Feuer,Feuer
5,GLUTEXO,58,64,58,80,80,65,Feuer,Feuer
...,...,...,...,...,...,...,...,...,...
147,DRATINI,41,64,45,50,50,50,Drache,Drache
148,DRAGONIR,61,84,65,70,70,70,Drache,Drache
149,DRAGORAN,91,134,95,80,100,100,Drache,Flug


In [42]:
df_moves = pd.read_csv("../data/moves.csv", index_col="move_id")

df_moves

,move_name,effekt,staerke,typ,genauigkeit,pp
move_id,,,,,,
1,PFUND,0,40,Normal,100,35
2,KARATESCHLAG,43,50,Kampf,100,25
3,DUPLEXHIEB,29,15,Normal,85,10
4,KOMETENHIEB,29,18,Normal,85,15
5,MEGAHIEB,0,80,Normal,85,20
...,...,...,...,...,...,...
161,TRIPLETTE,36,80,Normal,100,10
162,SUPERZAHN,40,1,Normal,90,10
163,SCHLITZER,43,70,Normal,100,20
